# Bulk revision sync

Fetch three months of PAYEMS revision history in one ALFRED request and store the batch in SQLite.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import display

from asof import SQLiteRevisionStore, sync_revision_histories

ENV_PATH = Path(find_dotenv())
DATABASE_PATH = ENV_PATH.parent / "data" / "asof.sqlite3"
load_dotenv(ENV_PATH)
API_KEY = os.getenv("FRED_API_KEY")
assert API_KEY, "FRED_API_KEY was not loaded from .env"

In [2]:
with SQLiteRevisionStore(DATABASE_PATH) as store:
    result = sync_revision_histories(
        store=store,
        series_id="PAYEMS",
        observation_start="2008-07-01",
        observation_end="2008-09-01",
        api_key=API_KEY,
    )

print(
    f"Stored {result.observations} observations and {result.vintages} vintages "
    f"for {result.series_id}."
)

Stored 3 observations and 51 vintages for PAYEMS.


In [3]:
with SQLiteRevisionStore(DATABASE_PATH) as store:
    observation_dates = store.list_observations(series_id="PAYEMS")
    stored = [
        store.load_history(series_id="PAYEMS", observation_date=observation_date)
        for observation_date in observation_dates
    ]

summary = pd.DataFrame(
    [
        {
            "observation_date": history.observation_date,
            "vintages": len(history.revisions),
            "initial_value": history.initial.value,
            "latest_value": history.current.value,
            "total_revision": history.total_revision,
        }
        for history in stored
    ]
)
display(summary.style.format(precision=1, thousands=","))

,observation_date,vintages,initial_value,latest_value,total_revision
0,2008-07-01,17,"137,615.0","137,497.0",-118.0
1,2008-08-01,16,"137,473.0","137,211.0",-262.0
2,2008-09-01,18,"137,318.0","136,760.0",-558.0
